# Notebook 12: Retrieval-Augmented Generation (RAG) & Conversational Recommendations

## KLEAN Chatbot — Restaurant Information Retrieval & Recommendation

### Overview
In Phases 1–6, we built and evaluated our Information Retrieval engines (Boolean, TF-IDF, BM25, FAISS Vector Search, and Hybrid RRF).

In **Phase 7**, we connect our top-performing **Hybrid Retrieval Engine** to a **Large Language Model (DeepSeek / OpenAI / Gemini / Ollama)** to build a fully conversational, grounded dining assistant.

### Core RAG Steps:
1. **Retrieve (R)**: User natural query $\rightarrow$ `HybridRetriever` fetches top-$k$ restaurant candidates ($0.5 \text{ BM25} + 0.5 \text{ Vector via RRF}$).
2. **Augment (A)**: `ContextBuilder` extracts verified metadata (Name, Category, Cuisine, Address, Contact) into a structured prompt with strict anti-hallucination guardrails.
3. **Generate (G)**: `LLMClient` generates a friendly, grounded, and bilingual response citing specific verified restaurants.

In [ ]:
import sys
import os
from pathlib import Path
from dotenv import load_dotenv

# Add project root to sys.path and load environment variables
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")

from src.utils.data_loader import DataLoader
from src.utils.document_converter import DocumentConverter
from src.preprocessing.text_preprocessing import TextPreprocessor
from src.indexing.inverted_index import InvertedIndex
from src.retrieval.bm25 import BM25Retriever
from src.embeddings.vector_store import VectorStore
from src.embeddings.embedding_model import EmbeddingModel
from src.retrieval.vector_search import VectorSearcher
from src.retrieval.hybrid import HybridRetriever
from src.rag.context_builder import ContextBuilder
from src.rag.prompt_templates import PromptManager
from src.rag.llm_client import get_llm_client
from src.rag.rag_pipeline import RAGPipeline

print(f"Project root resolved: {PROJECT_ROOT}")
print(f"Configured LLM Provider: {os.getenv('LLM_PROVIDER', 'mock')} (Model: {os.getenv('LLM_MODEL', 'default')})")

## 1. Initialize Hybrid Retriever and RAG Pipeline

In [ ]:
# Load 698 restaurants
loader = DataLoader(PROJECT_ROOT / "dataset" / "restaurants.json")
records = loader.load()
converter = DocumentConverter()
converter.convert_all(records)
corpus = converter.get_corpus()

# Build BM25
preprocessor = TextPreprocessor()
tokenized_corpus = preprocessor.preprocess_documents(corpus)
index = InvertedIndex()
index.build(tokenized_corpus)
bm25 = BM25Retriever(index, preprocessor)

# Load FAISS
idx_path = PROJECT_ROOT / "vector_db" / "faiss_index.bin"
meta_path = PROJECT_ROOT / "vector_db" / "index_metadata.json"
vector_store = VectorStore.load(idx_path, meta_path)
embedding_model = EmbeddingModel()
vector_searcher = VectorSearcher(vector_store=vector_store, embedding_model=embedding_model)

# Build Hybrid Retriever
hybrid = HybridRetriever(bm25_retriever=bm25, vector_searcher=vector_searcher)

# Build RAG Context & LLM Client
records_by_id = {r["id"]: r for r in records}
context_builder = ContextBuilder(records_by_id=records_by_id)
llm_client = get_llm_client(fallback_to_mock=True)
prompt_manager = PromptManager()

# End-to-End Pipeline
rag = RAGPipeline(
    retriever=hybrid,
    context_builder=context_builder,
    llm_client=llm_client,
    prompt_manager=prompt_manager,
    default_top_k=3,
)

print(f"RAGPipeline initialized successfully using LLM Client: {llm_client.__class__.__name__}")

## 2. Test Conversational Recommendations (English Queries)

In [ ]:
test_queries = [
    "Where can I get authentic Vietnamese pho noodle soup in Phnom Penh?",
    "Recommend a cozy coffee shop in Toul Kork where I can read or work.",
    "I'm looking for a casual dinner place with barbecue and cold beer.",
]

for q in test_queries:
    print("=" * 80)
    print(f"USER INQUIRY: {q}")
    print("=" * 80)
    
    res = rag.ask(q, top_k=3)
    print(f"\n[KLEAN CHATBOT RESPONSE] (Generated in {res['latency_ms']:.1f}ms):\n")
    print(res["answer"])
    
    print("\n--- Verified Sources Cited ---")
    for src in res["retrieved_restaurants"]:
        print(f"  • [ID {src['doc_id']}] {src['name']} | Address: {src['address']}")
    print("\n")

## 3. Bilingual Support: Khmer Language Inquiry

The system prompt instructs the model to detect and respond fluently in Khmer when the user communicates in Khmer.

In [ ]:
khmer_query = "តើខ្ញុំអាចរកញ៉ាំបាយសាច់ជ្រូក ឬគុយទាវឆ្ងាញ់នៅឯណា?"
print("=" * 80)
print(f"KHMER QUERY: {khmer_query}")
print("=" * 80)

res_khmer = rag.ask(khmer_query, top_k=3)
print(f"\n[KLEAN CHATBOT RESPONSE] ({res_khmer['latency_ms']:.1f}ms):\n")
print(res_khmer["answer"])

print("\n--- Verified Sources Cited ---")
for src in res_khmer["retrieved_restaurants"]:
    print(f"  • [ID {src['doc_id']}] {src['name']} | Address: {src['address']}")

## 4. Anti-Hallucination & Guardrail Verification

What happens when the user asks for something completely unavailable in our Phnom Penh database? (e.g. *traditional Icelandic fermented shark*).

A poorly designed RAG system might hallucinate a fake Icelandic restaurant. A well-designed RAG system detects the absence and politely informs the user.

In [ ]:
absent_query = "Where can I eat traditional Icelandic fermented shark and smoked puffin in Phnom Penh?"
print("=" * 80)
print(f"OUT-OF-DOMAIN QUERY: {absent_query}")
print("=" * 80)

res_guardrail = rag.ask(absent_query, top_k=3)
print(f"\n[KLEAN CHATBOT RESPONSE]:\n")
print(res_guardrail["answer"])

## 5. Latency Decomposition Analysis

We inspect where time is spent in the end-to-end RAG pipeline.

In [ ]:
sample_q = "casual pizza and pasta place in tonle bassac"
res_timed = rag.ask(sample_q, top_k=3)

print(f"Latency Decomposition for '{sample_q}':")
print(f"  1. Retrieval Time (BM25 + FAISS Inner Product) : {res_timed['retrieval_latency_ms']:.2f} ms")
print(f"  2. LLM Generation Time (API Network Call)     : {res_timed['llm_latency_ms']:.2f} ms")
print(f"  ------------------------------------------------------------")
print(f"  Total End-to-End Latency                       : {res_timed['latency_ms']:.2f} ms")

## 6. Key Takeaways

1. **Grounded Intelligence**: By piping real, verified restaurant data into the prompt, the LLM provides specific, trustworthy recommendations rather than generic advice.
2. **Strict Guardrails**: The prompt template successfully prevents hallucinations on nonexistent dishes or cuisines.
3. **Bilingual Fluency**: Seamlessly switches between English and Khmer depending on user language.
4. **Next Step (Phase 8)**: Package this entire RAG pipeline into a **FastAPI backend** with `/search` and `/chat` endpoints!